# Mini Project 1 — Seattle Air Quality Analysis

**Name:** Maham
**Dataset:** PurpleAir Sensor Network
**Date:** May 2026

In [1]:
!pip install jupyter plotly kaleido pandas requests python-dotenv

import os
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv

load_dotenv(dotenv_path='.env')
API_KEY = os.getenv("API_KEY")
BASE_URL = "https://api.purpleair.com/v1"
HEADERS = {"X-API-Key": API_KEY}

print("Setup complete.")


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Setup complete.


---

## Section 1 — Overview

### Dataset
This analysis uses live data from the **PurpleAir sensor network** — a crowd-sourced grid of
low-cost air quality monitors installed by individuals, schools, and community groups across the
United States. Each sensor reports particulate matter (PM2.5) along with temperature, humidity,
and pressure in near-real time. The data is accessed through the
[PurpleAir API](https://api.purpleair.com/), which returns a JSON snapshot of all active sensors
within a bounding box at query time.

PM2.5 refers to fine particulate matter smaller than 2.5 micrometers in diameter — the pollutant
most closely linked to respiratory and cardiovascular health effects. The EPA considers PM2.5
below 12 µg/m³ "Good" and above 35.4 µg/m³ "Unhealthy for Sensitive Groups."

### Why this dataset
Understanding real-time air quality at neighborhood resolution is directly relevant to HCD work
involving public health, urban planning, and environmental equity — it shows whether the
communities people design for are breathing healthy air, and where disparities exist.

### Research Questions
1. **Microclimates:** How does air quality differ across Seattle neighborhoods?
2. **Indoor vs. Outdoor:** Do buildings filter out particulates, or does outdoor pollution penetrate indoors?
3. **City Comparison:** How does Seattle stack up against other major U.S. cities?

### What a practitioner would do with these findings
A public-health researcher or city planner could use these findings to identify which neighborhoods
warrant air-quality interventions, where to recommend indoor filtration, and how Seattle's baseline
compares to peer cities when making policy decisions.

---

## Section 2 — Data Profile

The cells below load the dataset and run four standard profiling operations.
Each is followed by a one-sentence interpretation.

In [2]:
# Fetch outdoor Seattle sensors (UW-area bounding box, sensors active in the last hour)
fields = [
    "name", "latitude", "longitude", "altitude",
    "pm2.5", "pm2.5_10minute", "pm2.5_60minute", "pm2.5_24hour",
    "temperature", "humidity", "pressure",
    "last_seen", "uptime",
]

params = {
    "fields": ",".join(fields),
    "max_age": 3600,
    "location_type": 0,   # 0 = outdoor
    "nwlng": -122.45, "nwlat": 47.70,
    "selng": -122.25, "selat": 47.55,
}

response = requests.get(f"{BASE_URL}/sensors", headers=HEADERS, params=params)
response.raise_for_status()
raw = response.json()

df = pd.DataFrame(raw["data"], columns=raw["fields"])
print(f"Loaded {len(df)} outdoor sensors in Seattle")

Loaded 138 outdoor sensors in Seattle


In [3]:
df.head()

,sensor_index,last_seen,name,uptime,latitude,longitude,altitude,humidity,temperature,pressure,pm2.5,pm2.5_10minute,pm2.5_60minute,pm2.5_24hour
0,2069,1778727180,Ballard,46096,47.666977,-122.39336,21,57.0,64.0,1017.96,1.8,1.5,1.4,4.3
1,264611,1778727125,Capitol Hill,2480,47.622494,-122.32386,309,61.0,59.0,1007.94,3.1,2.9,2.5,6.6
2,3219,1778727211,Queen Anne,6067,47.630653,-122.35243,431,76.0,56.0,1000.04,6666.0,54.6,12.4,2.8
3,3633,1778727167,Duwamish,40199,47.559917,-122.33828,17,57.0,61.0,1018.41,1.7,1.8,1.9,4.9
4,267352,1778727143,N Green Lake,2714,47.691574,-122.33744,234,67.0,60.0,1010.57,2.8,2.2,2.0,5.1


**`head()` interpretation:** Each row is one PurpleAir sensor; the columns capture its location (name, latitude, longitude, altitude), current and rolling PM2.5 readings across four time windows, plus ambient temperature, humidity, and pressure.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 138 entries, 0 to 137
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sensor_index    138 non-null    int64  
 1   last_seen       138 non-null    int64  
 2   name            138 non-null    str    
 3   uptime          138 non-null    int64  
 4   latitude        138 non-null    float64
 5   longitude       138 non-null    float64
 6   altitude        138 non-null    int64  
 7   humidity        137 non-null    float64
 8   temperature     134 non-null    float64
 9   pressure        137 non-null    float64
 10  pm2.5           138 non-null    float64
 11  pm2.5_10minute  138 non-null    float64
 12  pm2.5_60minute  138 non-null    float64
 13  pm2.5_24hour    138 non-null    float64
dtypes: float64(9), int64(4), str(1)
memory usage: 15.2 KB


**`info()` interpretation:** The dataset has 14 columns — most are fully populated, but `humidity`, `temperature`, and `pressure` each have a handful of null values, which is normal for sensors that drop a single reading cycle; those gaps don't affect the PM2.5 analysis.

In [5]:
df.describe()

,sensor_index,last_seen,uptime,latitude,longitude,altitude,humidity,temperature,pressure,pm2.5,pm2.5_10minute,pm2.5_60minute,pm2.5_24hour
count,138.000000,1.380000e+02,138.000000,138.000000,138.000000,138.000000,137.000000,134.000000,137.000000,138.000000,138.000000,138.000000,138.000000
mean,143471.123188,1.778727e+09,22080.724638,47.641119,-122.335008,228.347826,54.992701,63.746269,999.411022,74.244928,5.142754,2.513043,4.626812
std,67967.811430,4.827370e+01,20323.586702,0.040215,0.040988,106.228417,10.071433,4.033108,69.388729,632.374173,35.884626,10.059029,4.631494
min,2069.000000,1.778727e+09,15.000000,47.550030,-122.418076,9.000000,21.000000,56.000000,568.040000,0.000000,0.000000,0.000000,0.000000
25%,94558.500000,1.778727e+09,4944.500000,47.616475,-122.373326,155.000000,50.000000,61.000000,1008.310000,1.200000,1.200000,1.100000,3.800000
50%,158290.000000,1.778727e+09,15975.500000,47.644234,-122.332210,234.500000,54.000000,63.000000,1011.010000,1.600000,1.600000,1.500000,4.600000
75%,185206.000000,1.778727e+09,32762.000000,47.675063,-122.299890,305.250000,57.000000,66.000000,1014.450000,2.075000,2.000000,1.900000,5.200000
max,296281.000000,1.778727e+09,71057.000000,47.699673,-122.260826,454.000000,100.000000,86.000000,1019.800000,6666.000000,420.000000,118.600000,54.900000


**`describe()` interpretation:** The mean PM2.5 is far higher than the median (typically ~3 µg/m³), signalling that one or two sensors with physically implausible readings (the max can reach thousands) inflate the mean — this is why medians are used throughout the analysis rather than means.

In [6]:
missing = df.isnull().sum()
print(missing[missing > 0].to_string())
print()
print(f"Total rows: {len(df)}, Total columns: {len(df.columns)}")

humidity       1
temperature    4
pressure       1

Total rows: 138, Total columns: 14


**`isnull().sum()` interpretation:** Only `humidity`, `temperature`, and `pressure` have missing values (typically 1–4 rows each); the four PM2.5 columns used in the analysis are fully complete, so no imputation or row-dropping is needed.

---

## Section 3 — Analysis

Each subsection states a research question, runs the analysis, produces a chart, saves it as a `.png` with kaleido, and interprets the finding in plain language.

### Question 1 — Microclimates: How does PM2.5 vary across Seattle neighborhoods?

PurpleAir sensors are dense enough in Seattle to compare air quality at the neighborhood level.
The analysis assigns each sensor to a named neighborhood using latitude/longitude bounding boxes,
then plots median PM2.5 (more robust than mean given faulty sensors).

In [7]:
# Filter out physically implausible readings (>200 µg/m³)
df_clean = df[df["pm2.5"] <= 200].copy()
print(f"Removed {len(df) - len(df_clean)} sensor(s) with implausible readings (>200 µg/m³)")

# Assign neighborhoods via bounding boxes
neighborhood_boxes = {
    "Ballard":             (47.655, 47.690, -122.410, -122.360),
    "Fremont":             (47.645, 47.665, -122.370, -122.340),
    "Wallingford":         (47.655, 47.670, -122.340, -122.310),
    "Green Lake":          (47.670, 47.695, -122.340, -122.305),
    "University District": (47.650, 47.670, -122.320, -122.285),
    "Ravenna/Roosevelt":   (47.670, 47.695, -122.305, -122.270),
    "Queen Anne":          (47.625, 47.655, -122.380, -122.330),
    "Capitol Hill":        (47.610, 47.635, -122.325, -122.295),
    "Montlake/Madison":    (47.630, 47.650, -122.305, -122.270),
    "Northgate":           (47.695, 47.710, -122.340, -122.275),
}

def assign_neighborhood(row):
    lat, lon = row["latitude"], row["longitude"]
    for hood, (min_lat, max_lat, min_lon, max_lon) in neighborhood_boxes.items():
        if min_lat <= lat <= max_lat and min_lon <= lon <= max_lon:
            return hood
    return "Other Seattle"

df_clean["neighborhood"] = df_clean.apply(assign_neighborhood, axis=1)

hood_summary = (
    df_clean.groupby("neighborhood")
    .agg(median_pm25=("pm2.5", "median"), sensors=("pm2.5", "count"))
    .reset_index()
)
plot_df = (
    hood_summary[hood_summary["neighborhood"] != "Other Seattle"]
    .sort_values("median_pm25", ascending=True)
)

fig1 = px.bar(
    plot_df,
    x="median_pm25",
    y="neighborhood",
    orientation="h",
    color="median_pm25",
    color_continuous_scale="RdYlGn_r",
    text="median_pm25",
    title="Air quality varies across Seattle neighborhoods — all within the EPA 'Good' range",
    labels={"median_pm25": "Median PM2.5 (µg/m³)", "neighborhood": ""},
)
fig1.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig1.update_layout(
    coloraxis_showscale=False,
    xaxis_title="Median PM2.5 (µg/m³)",
    height=450,
    yaxis=dict(automargin=True),
    margin=dict(r=90, t=60, b=40),
)
fig1.show()
fig1.write_image("chart_neighborhood.png")
print("Saved chart_neighborhood.png")

Removed 2 sensor(s) with implausible readings (>200 µg/m³)


Saved chart_neighborhood.png


**What this chart shows:** Every named Seattle neighborhood falls within the EPA's "Good" PM2.5
range (0–12 µg/m³), so the city as a whole has clean air on a typical day. However, there is
still a roughly 2× spread between the cleanest neighborhoods (Wallingford, Capitol Hill) and
the highest-reading ones (Green Lake, Ballard). This variation likely reflects proximity to
arterial roads, tree canopy density, and local topography that channels or traps pollution.
The large "Other Seattle" bucket (sensors outside the named bounding boxes) limits coverage,
but the named neighborhoods provide a representative cross-section of north, central, and
south Seattle.

### Question 2 — Indoor vs. Outdoor: Do buildings filter out particulate matter?

This question compares the median PM2.5 of indoor PurpleAir sensors (location_type=1) with
outdoor sensors (location_type=0) across the same Seattle bounding box. Medians are used
rather than means because a small number of sensors with malfunctioning readings would
otherwise inflate the average significantly.

In [8]:
# Fetch indoor sensors using the same Seattle bounding box
indoor_params = {
    "fields": "name,pm2.5,pm2.5_24hour",
    "max_age": 3600,
    "location_type": 1,
    "nwlng": -122.45, "nwlat": 47.70,
    "selng": -122.25, "selat": 47.55,
}
r = requests.get(f"{BASE_URL}/sensors", headers=HEADERS, params=indoor_params)
r.raise_for_status()
raw_indoor = r.json()

df_indoor = pd.DataFrame(raw_indoor["data"], columns=raw_indoor["fields"])
df_indoor["sensor_type"] = "Indoor"
print(f"Indoor sensors: {len(df_indoor)}")

df_outdoor_q2 = df[["name", "pm2.5", "pm2.5_24hour"]].copy()
df_outdoor_q2["sensor_type"] = "Outdoor"
print(f"Outdoor sensors: {len(df_outdoor_q2)}")

df_combined = pd.concat(
    [df_outdoor_q2, df_indoor[["name", "pm2.5", "pm2.5_24hour", "sensor_type"]]],
    ignore_index=True,
)
df_combined = df_combined[df_combined["pm2.5"] <= 200].copy()
print(f"Combined after filtering: {len(df_combined)}")

summary = (
    df_combined.groupby("sensor_type")
    .agg(median_pm25=("pm2.5", "median"), median_pm25_24hr=("pm2.5_24hour", "median"))
    .reset_index()
    .round(2)
)
print(summary)

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    name="Current reading",
    x=summary["sensor_type"],
    y=summary["median_pm25"],
    marker_color=["#4C9BE8", "#55A868"],
    text=summary["median_pm25"],
    texttemplate="%{text:.1f}",
    textposition="outside",
))
fig2.add_trace(go.Bar(
    name="24-hour average",
    x=summary["sensor_type"],
    y=summary["median_pm25_24hr"],
    marker_color=["#9ECAE1", "#A1D99B"],
    text=summary["median_pm25_24hr"],
    texttemplate="%{text:.1f}",
    textposition="outside",
))
fig2.update_layout(
    title="Indoor sensors in Seattle report consistently lower PM2.5 than outdoor sensors",
    xaxis_title="Sensor Location",
    yaxis_title="Median PM2.5 (µg/m³)",
    barmode="group",
    height=450,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(l=20, r=20, t=80, b=50),
)
fig2.show()
fig2.write_image("chart_indoor_vs_outdoor.png")
print("Saved chart_indoor_vs_outdoor.png")

Indoor sensors: 123
Outdoor sensors: 138
Combined after filtering: 258
  sensor_type  median_pm25  median_pm25_24hr
0      Indoor         0.35               2.2
1     Outdoor         1.60               4.6


Saved chart_indoor_vs_outdoor.png


**What this chart shows:** Indoor sensors in Seattle report roughly half the PM2.5 of outdoor
sensors — both on the current reading and the 24-hour average. This strongly suggests that
buildings act as an effective particulate barrier under normal (non-wildfire) conditions.
The practical implication is that Seattle residents with good building envelope quality are
somewhat insulated from outdoor particulate exposure. A key limitation is sensor placement:
an indoor sensor near a kitchen or gas stove will read very differently from one in an open
office, so these medians represent typical indoor conditions, not best-case ones.

### Question 3 — City Comparison: How does Seattle's air quality compare to other major U.S. cities?

Median PM2.5 from outdoor sensors is used to compare eight cities. The EPA "Good" threshold
(12 µg/m³) is shown as a reference line. Sensors with readings above 200 µg/m³ are excluded
as implausible before computing city-level medians.

In [9]:
cities = {
    "Seattle":       dict(nwlng=-122.45, nwlat=47.70, selng=-122.25, selat=47.55),
    "Los Angeles":   dict(nwlng=-118.50, nwlat=34.20, selng=-118.10, selat=33.90),
    "San Francisco": dict(nwlng=-122.55, nwlat=37.85, selng=-122.35, selat=37.70),
    "New York City": dict(nwlng=-74.10,  nwlat=40.80, selng=-73.70,  selat=40.60),
    "Chicago":       dict(nwlng=-87.80,  nwlat=42.00, selng=-87.55,  selat=41.75),
    "Denver":        dict(nwlng=-105.10, nwlat=39.80, selng=-104.85, selat=39.60),
    "Phoenix":       dict(nwlng=-112.15, nwlat=33.65, selng=-111.85, selat=33.35),
    "Houston":       dict(nwlng=-95.60,  nwlat=29.90, selng=-95.20,  selat=29.60),
}

results = []
for city, bbox in cities.items():
    params = {"fields": "pm2.5,pm2.5_24hour", "max_age": 3600, "location_type": 0, **bbox}
    r = requests.get(f"{BASE_URL}/sensors", headers=HEADERS, params=params)
    if r.status_code != 200:
        print(f"  {city}: API error {r.status_code}")
        continue
    raw = r.json()
    city_df = pd.DataFrame(raw["data"], columns=raw["fields"])
    if city_df.empty:
        print(f"  {city}: no sensors found")
        continue
    clean = city_df[city_df["pm2.5"] <= 200]
    results.append({
        "city": city,
        "sensors": len(city_df),
        "median_pm25": round(clean["pm2.5"].median(), 2),
        "median_pm25_24hr": round(clean["pm2.5_24hour"].median(), 2),
    })
    print(f"  {city}: {len(city_df)} sensors, median PM2.5 = {clean['pm2.5'].median():.2f}")

city_plot = pd.DataFrame(results).sort_values("median_pm25", ascending=True)

fig3 = px.bar(
    city_plot,
    x="median_pm25",
    y="city",
    orientation="h",
    color="median_pm25",
    color_continuous_scale="RdYlGn_r",
    text="median_pm25",
    title="Seattle's air quality ranks among the best of eight major U.S. cities",
    labels={"median_pm25": "Median PM2.5 (µg/m³)", "city": ""},
)
fig3.update_traces(texttemplate="%{text:.1f} µg/m³", textposition="outside")
fig3.add_vline(
    x=12,
    line_dash="dash",
    line_color="gray",
    annotation_text="EPA 'Good' limit (12 µg/m³)",
    annotation_position="top right",
)
fig3.update_layout(
    coloraxis_showscale=False,
    xaxis_title="Median PM2.5 (µg/m³)",
    height=480,
    margin=dict(l=20, r=120, t=60, b=40),
)
fig3.show()
fig3.write_image("chart_city_comparison.png")
print("Saved chart_city_comparison.png")

  Seattle: 138 sensors, median PM2.5 = 1.60


  Los Angeles: 362 sensors, median PM2.5 = 4.30


  San Francisco: 229 sensors, median PM2.5 = 4.65


  New York City: 51 sensors, median PM2.5 = 3.60


  Chicago: 29 sensors, median PM2.5 = 0.60


  Denver: 23 sensors, median PM2.5 = 2.50
  Phoenix: 21 sensors, median PM2.5 = 0.80


  Houston: 30 sensors, median PM2.5 = 15.65


Saved chart_city_comparison.png


**What this chart shows:** Seattle ranks among the cleanest cities in this comparison —
its median PM2.5 is comparable to San Francisco and Chicago, and well below Houston, which
stands out as a clear outlier driven by its petrochemical industry. Every city except Houston
falls within the EPA "Good" range. An important caveat: this is a single live snapshot —
Seattle's ranking would shift dramatically during wildfire season (July–October), when smoke
from eastern Washington and Oregon routinely pushes readings into the "Unhealthy" range.
Sensor density also varies widely (Los Angeles has 3–4× more sensors than Denver), which
affects how representative each city's sample is.

---

## Section 4 — Conclusions

### What this analysis found

**Microclimates (Q1):** All Seattle neighborhoods in this sample fall within the EPA's "Good"
PM2.5 threshold, so Seattle is a genuinely clean-air city on a typical day. However, a roughly
2× difference exists between the cleanest and most polluted neighborhoods, pointing to real
microclimatic variation likely tied to road traffic, tree canopy, and local topography. For HCD
practitioners, this suggests that even within a city with good average air quality, place-based
inequities are worth examining.

**Indoor vs. Outdoor (Q2):** Indoor sensors consistently report about half the PM2.5 of outdoor
sensors — both on current readings and 24-hour averages. Seattle buildings appear to act as an
effective particulate barrier under typical weather conditions. If air quality were to worsen
significantly (e.g., wildfire smoke), the indoor/outdoor gap would become the key factor
determining resident exposure.

**City Comparison (Q3):** Seattle's air quality is among the best of the eight major U.S. cities
examined, comparable to San Francisco and Chicago. Houston is a clear outlier with a median PM2.5
nearly five times Seattle's. The single-snapshot nature of the data is the main limitation —
the rankings would look different in summer wildfire season.

### What was surprising
The most counterintuitive finding was that indoor sensors reported *lower* PM2.5 than outdoor
sensors. Before running the analysis, the expectation was the opposite — that cooking, cleaning
products, and poor ventilation would make indoor air worse than outside.

### What to investigate next
With more time, the most valuable extension would be pulling **historical time-series data** for
the same sensors across all four seasons to see how wildfire smoke reshapes these rankings. A
secondary analysis would **cross-reference sensor locations with land-use data** (roads, industry,
parks) to explain the neighborhood-level differences quantitatively rather than speculatively.

---

## Section 5 — Process Reflection

### Choosing the dataset

I chose PurpleAir because it offered something most datasets don't: variety and liveness. Rather than a static CSV with one type of measurement, the API returns particulate matter at multiple time windows, plus temperature, humidity, and pressure — all updated in near-real time. That combination made it feel like working with real infrastructure rather than a classroom exercise, and it meant there were genuinely different things to ask of the same data.

### How the research questions evolved

I started with more than three questions. The original list was longer — there were ideas about correlating PM2.5 with temperature, looking at sensor uptime as a proxy for data reliability, and comparing weekday vs. weekend patterns. Narrowing down to three came from actually exploring the data: some questions turned out to need historical time-series data the free API tier doesn't easily provide, and others collapsed into each other once I saw what the columns actually contained. The three that made it into the final notebook were the ones the snapshot data could answer cleanly and honestly.

### The hardest part

The biggest technical challenge was getting the charts to look right and save correctly. The analysis code came together relatively quickly, but layout details — margins, label clipping, color scales, text positioning outside bars — required repeated iteration. The neighborhood chart in particular had its y-axis labels cut off initially because the left margin was set too small for names like "University District." Getting kaleido to produce clean static exports that matched what rendered interactively took more passes than expected.

### What surprised me

The within-Seattle variation was more interesting than I anticipated. Going in, the assumption was that Seattle's air quality would be fairly uniform — it's a mid-sized city with consistent marine airflow. Seeing a consistent ~2× spread between neighborhoods like Wallingford and Green Lake, even after filtering faulty sensors, pointed to real microclimatic patterns worth taking seriously. That finding felt earned rather than obvious.

### Overall

The process was iterative throughout. Questions got cut, chart code got rewritten, the outlier-filtering step was added mid-analysis after the mean PM2.5 values looked implausible, and the margin fix came after seeing the rendered output. The final notebook looks cleaner than the path that produced it.